In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from HHO_kernel import HHO_kernel, HHO_cell

plt.rcParams.update({
    # Make tick labels point inside
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    # Set thicker axis lines (spines)
    'axes.linewidth': 1.5,
})

# Define a prototypical cell

In [ ]:
k = 5 # cell degree
xL = 2.0 # left boundary
xR = 2.5 # right boundary
pcell = HHO_cell(xL,xR,k)
xx = np.linspace(xL,xR,50) # standard grid for visualizations

# Check the computation of the monomial basis functions

In [ ]:
# Evaluate basis on the prototypical cell
basis, gradient = pcell.evaluate_basis(xx)

# Plot the results
fig, axs = plt.subplots(1, 2,
                        figsize=(10,5))
for kk in range(k+1):
    label = rf"$\phi_{kk} \sim x^{kk}$"
    axs[0].plot(xx, basis[:,kk], label=label)
    axs[1].plot(xx, gradient[:,kk])
for ax in axs:
    ax.tick_params(direction='in')
    ax.set_xlabel('x')
axs[0].set_title("Basis functions")
axs[1].set_title("Derivative")
axs[0].set_ylabel(r"$\phi(x)$")
axs[1].set_ylabel(r"$\phi'(x)$")
axs[0].legend()
fig.suptitle("Monomial basis");


# Validation of the implementation of the L2-projection

We compute the mass matrix for the case of a low polynomial degree for analytical validation (k=2).

In [ ]:
# Define test example
cell_test_mass = HHO_cell(-1,1,2)
print("Computed mass matrix:")
print(np.round(cell_test_mass.mass_matrix,3))
# Compute the actual mass matrix from the computed cholesky decomposition
# (not to be used in the kernel since matrix multiplication is unstable)
mass = cell_test_mass.mass_cho
lower = cell_test_mass._mass_cho_lower
if lower:
    mass = np.tril(mass) @ np.tril(mass).T 
else:
    mass = np.triu(mass).T @ np.triu(mass)
print("Reconstructed mass matrix from Cholesky decomposition:")
print(np.round(mass,3))

This is exactly the expected result (up to computer precision).
Next we show the projection of some polynomials and a cosine function.

In [ ]:
# Define a routine to test several stages of the implementation of the HHO method on the prototypical cell.
def test_HHO_computation(computation, pcell):
    # Define test functions for illustration
    f = [lambda x: (x-xL)**5 - (x-xL)*(x-xR),
         lambda x: x*(x-xL)**5 - (x-xL)*(x-xR),
         lambda x: np.cos(1.8*np.pi/(xR-xL)*(x-xL))]
    neg_ddf = [lambda x: -20*(x-xL)**3 + 2,
               lambda x: -20*x*(x-xL)**3 + 2,
               lambda x: (2*np.pi/(xR-xL))**2 * np.cos(2*np.pi/(xR-xL)*(x-xL))]
    f_titles = ["5th order polynomial",
                "6th order polynomial",
                "Cosine"]
    
    # Function to compute an HHO result of a function f and plot it on the axis ax along with the exact function f
    def plot_HHO_computation(f, neg_ddf, ax, computation):
        # Compute HHO result according to required input
        match computation:
            case "L2 projection":
                f_HHO, _ = pcell.evaluate_fun(xx, pcell.compute_L2_projection(f))
            case "Reconstruction":
                dofs = np.concatenate((pcell.compute_L2_projection(f), [f(xL), f(xR)]))
                f_HHO, _ = pcell.evaluate_fun(xx, pcell.compute_reconstruction(dofs), pcell.degree_reconstruction)
            case "Cell problem":
                pcell.solve(neg_ddf, f(xL), f(xR))
                dofs = np.concatenate((pcell.solution, pcell.solution_faces))
                f_HHO, _ = pcell.evaluate_fun(xx, pcell.compute_reconstruction(dofs), pcell.degree_reconstruction)
        # Plot f
        ax.plot(xx,f(xx),'r.',label="f")
        # Compute error in the max norm
        err_max = np.max(np.abs( f(xx) - f_HHO ))
        # Plot the projection on the cell
        label = f"{computation}\nMax error: {err_max:.1e}"
        ax.plot(xx,f_HHO,'b-',label=label)
        # Compute max error at the faces
        err_max_faces = np.max(np.abs(([f(xx[0]) - f_HHO[0], f(xx[-1]) - f_HHO[-1]])))
        print(f"Max error at the faces: {err_max_faces:.1e} ({f_titles[i]})")

    fig, axs = plt.subplots(len(f), 1 ,
                            figsize=(5,3.5*len(f)),
                            gridspec_kw={'hspace': 0.4})
    for i,ax in enumerate(axs):    
        plot_HHO_computation(f[i], neg_ddf[i], ax, computation)
        ax.legend(loc='center left',
                bbox_to_anchor=(1, 0.5))
        ax.tick_params(direction='in')
        ax.grid(True, 
                which='major', 
                linestyle='--', 
                linewidth=0.5, 
                color='lightgrey')
        axs[i].set_title(f_titles[i])
    
    fig.suptitle(f"Example of {computation}s")

test_HHO_computation("L2 projection", pcell)

As expected, the projection of a 5th order polynomial is exact on the prototypical cell with polynomial unknowns of degree 5. 
Yet, as is shown by the illustration for the 6th order polynomial, this is no longer true if we go beyond the polynomial order of the cell unknowns. 

### Convergence

In general, the projection error of any sufficiently regular function $f$ should satisfy
$$
\lVert f - \Pi^k_H(f) \rVert_{L^2(\Omega)} \leq C h^{k+1},
$$
and
$$
\lVert f - \Pi^k_H(f) \rVert_{H^1(\Omega)} \leq C h^{k},
$$
where $h$ is the grid spacing and $k$ the polynomial degree and the $H^1$-norm understood in the broken sense. This property is validated below with $f(x) = \cos(4\pi x)$ and $\Omega = (0,1)$.

In [ ]:
# We define here a function that shows convergence orders by plotting the L2 and H1 errors for different polynomial degrees.
# The same function is used to handle the L2 projection as well as the reconstruction that is studied below.
def test_HHO_convergence(computation):
    x_left = 0
    x_right = 1
    N_cells = 8*2**np.arange(0,6)
    f = lambda x : np.cos(4*np.pi*x)
    df = lambda x : -4*np.pi*np.sin(4*np.pi*x)
    neg_ddf = lambda x : (4*np.pi)**2 * np.cos(4*np.pi*x)
    test_degrees = [1, 2, 5, 6]

    fig, axs = plt.subplots(2, len(test_degrees), 
                            figsize=(12,8),
                            gridspec_kw={'hspace': 0.3})
    # Compute L2 and H1 errors for all test degrees
    for (ik, k_test) in enumerate(test_degrees):
        errors_L2 = np.zeros(len(N_cells))
        errors_H1 = np.zeros(len(N_cells))
        for (i,N) in enumerate(N_cells):
            grid_N = np.linspace(x_left, x_right, N+1)
            solver = HHO_kernel(grid_N, k_test)
            error_L2_norm = 0
            error_H1_norm = 0
            for c in solver.cells:
                # Define quadrature to compute the error
                quad_error_x, quad_error_w = c.quadrature((c.degree+2))
                # Compute requested HHO operation
                match computation:
                    case "L2 projection":
                        f_basis, f_gradient = c.evaluate_fun(quad_error_x, c.compute_L2_projection(f), c.degree)
                    case "Reconstruction":
                        # Compute DOFs of f in the discrete HHO space on the cell
                        dofs = np.concatenate((c.compute_L2_projection(f), [f(c.x_left), f(c.x_right)]))
                        f_basis, f_gradient = c.evaluate_fun(quad_error_x, c.compute_reconstruction(dofs), c.degree_reconstruction)
                # Compute norm of error
                error_L2_x = f_basis - f(quad_error_x)
                error_H1_x = f_gradient - df(quad_error_x)
                error_L2_norm += np.dot(quad_error_w, error_L2_x**2)
                error_H1_norm += np.dot(quad_error_w, error_H1_x**2) + error_L2_norm
            errors_L2[i] = np.sqrt(error_L2_norm)
            errors_H1[i] = np.sqrt(error_H1_norm)
        # Plot L2 and H1 errors for given test degree
        for inorm, norm in enumerate(["L2", "H1"]):
            ax = axs[inorm][ik]
            match norm:
                case "L2":
                    errors = errors_L2
                case "H1":
                    errors = errors_H1
            match computation:
                case "L2 projection":
                    match norm:
                        case "L2":
                            convergence_order_theory = k_test+1
                            label = r"$\|| f - \Pi^k_H(f) ||_{L^2(\Omega)}$"
                        case "H1":
                            convergence_order_theory = k_test
                            label = r"$\|| f - \Pi^k_H(f) ||_{H^1(\Omega)}$"
                case "Reconstruction":
                    match norm:
                        case "L2":
                            convergence_order_theory = k_test+2
                            label = r"$\|| u - R_H(\hat{\Pi}^k_H u) ||_{L^2(\Omega)}$"
                        case "H1":
                            convergence_order_theory = k_test+1
                            label = r"$\|| u - R_H(\hat{\Pi}^k_H u) ||_{H^1(\Omega)}$"
            # Plot errors
            ax.plot(1.0/N_cells, errors, '*-', linewidth=0.8, label=label)
            # Compute straight line that indicates the theoretical convergence order
            convergence_line = (1.0/N_cells)**(convergence_order_theory)
            # Rescale the straight line to ligh slightly above the first point of the errors
            convergence_line = convergence_line * 6 / convergence_line[0] * errors[0]
            # Plot layout
            ax.plot(1.0/N_cells, 
                    convergence_line, 
                    label=fr"$Ch^{convergence_order_theory}$")
            ax.set_xscale('log')
            ax.set_yscale('log')
            ax.set_xlabel('$h$')
            ax.grid(True, 
                        which='major', 
                        linestyle='--', 
                        linewidth=0.5, 
                        color='lightgrey')
            ax.set_title(f"k={k_test}")
            ax.legend(loc='upper left')
    fig.suptitle(f"Convergence rate of {computation}")

test_HHO_convergence("L2 projection")

The order of convergence matches the theoretical prediction until the mesh is so fine that the error is dominated by rounding errors.

# Validation of the implementation of the HHO reconstruction

We first need to compute the stiffness matrix for the higher-order reconstruction space, in which we neglect the constant basis function to avoid problems of well-posedness in the computation of the reconstruction. 
We check the value of the stiffness matrix on a small example. In fact, we use the same example as the one used for the stiffness matrix above.

In [ ]:
print("Computed stiffness matrix for the reconstruction problem:")
print(np.round(cell_test_mass.reconstruction_matrix,3))

This is exactly the expected stiffness matrix, as can be confirmed by analytical integrations.
We next show some examples of computed reconstructions on the prototypical cell.

In [ ]:
test_HHO_computation("Reconstruction", pcell)

In contrast to the $L^2$-projection, the reconstruction is exact even for the 6th order polynomial with cell degree 5 on the prototypical cell.
Indeed, the reconstruction is the elliptic projection on the higher-order space of all polynomials of degree 6.

Another theoretical property of the reconstruction operator (in 1D and for $k \geq 1$) is that the reconstruction matches the function $f$ to be reconstructed at the faces (vertices) of the cell. This can indeed be confirmed from the cosine example, which does not allow for an exact representation by the reconstruction, but the error at the faces is only of order $10^{-15}$.

### Convergence

For any function $u \in L^2(\Omega)$, let $\hat{\Pi}^k_H u$ denote the interpolate of $u$ in the discrete HHO space.
The HHO theory proves the following error estimates for any sufficiently regular function $u$:
$$
\lVert u - R_H(\hat{\Pi}^k_H u) \rVert_{L^2(\Omega)} \leq C h^{k+2},
$$
and
$$
\lVert u - R_H(\hat{\Pi}^k_H u) \rVert_{H^1(\Omega)} \leq C h^{k+1}.
$$
Note that, for $k \geq 1$, the reconstruction is continuous and the $H^1$-norm is not a broken norm in that case.
Like the convergence plots of the projection, the next results are based on the test case $u(x) = \cos(4\pi x)$.

In [ ]:
test_HHO_convergence("Reconstruction")

The convergence rate matches the theoretically predicted value for all degrees and in both the $L^2$- and $H^1$-norm. The highest-degree approximations again suffer from the accumulation of round-off errors when the approximation is already small.

# Stabilization
Since the reconstruction is continuous for $k \geq 1$, the stabilization operator vanishes. For $k=0$, we avoid the computation of the stabilization operator by implementing an explicit formula for the reconstruction. As a result, there is no stabilization operator to be computed in order to formulate the local problems for the reconstructed potential.

Below, we validate the implementation of the cell problems and the convergence of the HHO method as a whole for $k \geq 1$.

We should compare exactness for the two different forms of stabilization and also check that the stabilization matches the face unknowns at the faces (regardless of exactness).

# Cell problems

In [ ]:
test_HHO_computation("Cell problem", pcell)